In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [18]:
import os
os.getcwd()
os.listdir('notebook')

['data']

In [19]:
data = pd.read_csv('notebook/data/churn2.csv')

In [20]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [21]:
data.tail()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1
9999,10000,15628319,Walker,792,France,Female,28,4,130142.79,1,1,0,38190.78,0


#### Feature Engineering

In [22]:
data['has_zero_balance'] = (data['Balance'] == 0).astype(int)
bins = [17,30,40,50,60,92]
print(bins)
data['age_group'] = pd.cut(data['Age'], bins=bins, labels=False)
print(data['age_group'].isna().sum())
data['balance_salary_ratio'] = data['Balance'] / (data['EstimatedSalary'] + 1)

[17, 30, 40, 50, 60, 92]
0


In [23]:
data = pd.get_dummies(data, columns=['Geography', 'Gender'], drop_first=True)

In [24]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [25]:
data.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,has_zero_balance,age_group,balance_salary_ratio,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,1,2,0.000000,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,0,2,0.744670,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,0,2,1.401362,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,1,1,0.000000,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,0,2,1.587035,False,True,False


In [26]:
data.isna().sum()[data.isna().sum() > 0]

Series([], dtype: int64)

In [27]:
print(data['Age'].min(), data['Age'].max())

18 92


#### Train/test/split - Separating Features from Target

In [28]:
X = data.drop(columns=['Exited'])
y = data['Exited']

In [29]:
print(X)

      CreditScore  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0             619   42       2       0.00              1          1   
1             608   41       1   83807.86              1          0   
2             502   42       8  159660.80              3          1   
3             699   39       1       0.00              2          0   
4             850   43       2  125510.82              1          1   
...           ...  ...     ...        ...            ...        ...   
9995          771   39       5       0.00              2          1   
9996          516   35      10   57369.61              1          1   
9997          709   36       7       0.00              1          0   
9998          772   42       3   75075.31              2          1   
9999          792   28       4  130142.79              1          1   

      IsActiveMember  EstimatedSalary  has_zero_balance  age_group  \
0                  1        101348.88                 1          2   
1      

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

#### Baseline Model

In [31]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

#### Tree-based Model

In [33]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    scale_pos_weight=(y_train==0).sum() / (y_train==1).sum(),
    eval_metrics='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)

C:\GLASSIMB\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [15:06:42] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "eval_metrics" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None,
              eval_metrics='logloss', feature_types=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

#### Evaluation

In [36]:
for name, model, X_te in [('LogReg', lr, X_test_scaled), ('XGB', xgb, X_test)]:
    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)[:,1]
    print(f"--- {name} ---")
    print(classification_report(y_test, preds))
    print("ROC-AUC:", roc_auc_score(y_test, probs))

--- LogReg ---
              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1593
           1       0.39      0.71      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000

ROC-AUC: 0.7777207099240997
--- XGB ---
              precision    recall  f1-score   support

           0       0.90      0.89      0.89      1593
           1       0.57      0.60      0.59       407

    accuracy                           0.83      2000
   macro avg       0.73      0.74      0.74      2000
weighted avg       0.83      0.83      0.83      2000

ROC-AUC: 0.8306488306488307


#### Feature Importance

In [37]:
import pandas as pd
importances = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10))

NumOfProducts           0.252004
IsActiveMember          0.116615
Age                     0.114016
has_zero_balance        0.108545
Geography_Germany       0.080811
Balance                 0.055292
balance_salary_ratio    0.047851
Gender_Male             0.047159
Geography_Spain         0.045653
EstimatedSalary         0.036153
dtype: float32
